# Phase 1 : ancre BART-large (MEDIQA 2021)

Runtime : **GPU L4** (ou A100). Durée : environ 45 à 60 minutes. Toute la logique est dans le dépôt ; ce notebook ne fait que l'appeler.

1. Recherche du taux d'apprentissage (seed 0, choix sur le ROUGE-L du dev)
2. Seeds 1 et 2 au taux retenu, prédictions du dev (greedy et beam4)
3. Tableau du dev (moyenne ± écart-type sur 3 seeds)
4. **Un seul accès journalisé au test**, pour la reproduction de BART, comparée à FocusMed (ROUGE-L 0,293)
5. Archive des artefacts à télécharger (prédictions, scores, manifestes, registre du test)

Les modèles ne sont pas conservés (stockage éphémère) : seuls les artefacts légers comptent pour la suite.

In [ ]:
REPO_URL = "https://github.com/desire-del/faithful-question-summarization.git"
REF = "main"
%cd /content
!rm -rf repo && git clone -q --depth 1 --branch $REF $REPO_URL repo
%cd /content/repo
# Exact versions from uv.lock, except the torch stack (Colab's build matches its CUDA driver).
!pip install -q uv
!uv export --frozen --extra train --extra eval --no-hashes --no-emit-project -o /content/req_lock.txt
!grep -v -E '^(torch|triton|nvidia-|cuda-|torchvision|torchaudio)' /content/req_lock.txt > /content/req.txt
!pip uninstall -y -q torchao
!pip install -q -r /content/req.txt
!pip install -q -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Rebuild the splits and verify them against the versioned manifest
!faithqs data fetch

In [ ]:
# 1) Learning-rate search (seed 0), choice on dev ROUGE-L
!python scripts/run_sft_protocol.py --env colab --model bart_large --sft-config configs/sft/bart_large.yaml --tag bart --stage search

In [ ]:
# 2) Seeds 1 and 2 at the chosen rate + dev predictions (greedy, beam4)
!python scripts/run_sft_protocol.py --env colab --model bart_large --sft-config configs/sft/bart_large.yaml --tag bart --stage seeds

In [ ]:
# 3) Dev table
!python scripts/run_sft_protocol.py --env colab --model bart_large --sft-config configs/sft/bart_large.yaml --tag bart --stage table

## ⚠️ Accès au test, une seule fois
La cellule suivante consomme l'accès `bart_reproduction` prévu par le protocole (PROTOCOL §2). Elle est journalisée dans `reports/test_ledger.jsonl` puis fermée. **Ne la relance pas** : une seconde tentative est refusée.

In [ ]:
# 4) BART reproduction on the official MEDIQA 2021 test set (logged access)
!python scripts/run_sft_protocol.py --env colab --model bart_large --sft-config configs/sft/bart_large.yaml --tag bart --stage test

In [ ]:
# 5) Artifacts: run manifests, predictions, scores, reports and the test ledger (no model weights)
import pathlib, zipfile
runs = pathlib.Path('/content/faithqs-runs')
archive = pathlib.Path('/content/bart_anchor_artifacts.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in runs.rglob('*'):
        if path.is_file() and not {'model', 'checkpoints'} & set(path.parts):
            z.write(path, 'runs/' + str(path.relative_to(runs)))
    for path in pathlib.Path('reports').rglob('*'):
        if path.is_file():
            z.write(path, str(path))
print(archive, round(archive.stat().st_size / 1e6, 2), 'MB')
from google.colab import files
files.download(str(archive))

In [ ]:
# 6) Summary to paste back
import json
print(open('reports/sft/bart_dev.md').read())
rep = json.load(open('reports/sft/bart_test_reproduction.json'))
print('chosen lr', rep['lr'])
print(json.dumps(rep['vs_focusmed_bart'], indent=1))